In [ ]:
from pathlib import Path
import sys
import os
import ast
import shutil
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
BASE = REPO / "exp/eft_sp_piloto_v1"
CORRECAO = REPO / "exp/eft_tiny_2000_v2/eft_alinhamento_seguro.py"
PACOTE = REPO / "exp/eft_penultima_2000_v1"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), sys.executable

os.chdir(REPO)

def trocar(texto, antes, depois):
    assert texto.count(antes) == 1, (
        f"Fonte diferente do esperado: {antes}"
    )
    return texto.replace(antes, depois, 1)

modelo = (BASE / "eft_modelo.py").read_text()

modelo = trocar(
    modelo,
    'canais = variante_cfg["dims"] * 8',
    'canais = variante_cfg["dims"] * 4',
)

modelo = trocar(
    modelo,
    """            features = self.encoder(imagens)[-1]
            assert features.shape[1:] == (canais, 8, 8)""",
    """            features = self.encoder.patch_embed(imagens)
            for i in range(3):
                features = self.encoder.layers[i].blocks(features)
                if i < 2:
                    features = self.encoder.layers[i].downsample(features)

            assert self.encoder.channel_first
            assert features.shape[1:] == (canais, 16, 16)

            features = F.layer_norm(
                features.permute(0,2,3,1), (canais,),
                eps=self.encoder.outnorm3.eps,
            ).permute(0,3,1,2).contiguous()""",
)

modelo = trocar(
    modelo,
    "    modelo.train().requires_grad_(True)",
    """    modelo.train().requires_grad_(True)
    # Módulos posteriores à saída escolhida ficam inativos.
    modelo.encoder.layers[2].downsample.requires_grad_(False)
    modelo.encoder.layers[3].requires_grad_(False)
    modelo.encoder.outnorm3.requires_grad_(False)""",
)

treino = (BASE / "eft_treino.py").read_text()

alteracoes = [
    (
        "import eft_modelo",
        "import eft_modelo\n"
        "import eft_alinhamento_seguro\n"
        "from eft_alinhamento_seguro import instalar",
    ),
    (
        "    assert len(planos['treino']) == 200 and planos['validacao']",
        """
    origem_plano = repo / (
        'reproducao/eft_tiny_2000_v2_20261011T204725195685Z/planos.json'
    )
    fixo = json.loads(origem_plano.read_text())
    assert fixo['treino'][:200] == planos['treino']
    assert fixo['validacao'] == planos['validacao']
    planos['treino'] = fixo['treino']

    assert len(planos['treino']) == 2000
    assert len(planos['validacao']) == 145
    assert variante == 'tiny'

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.use_deterministic_algorithms(False)
""",
    ),
    (
        "assert len(ti) == 200 and not ti & vi",
        "assert len(ti) == 1304 and not ti & vi",
    ),
    (
        "__file__, eft_dados.__file__, eft_modelo.__file__,",
        "__file__, eft_dados.__file__, eft_modelo.__file__, "
        "eft_alinhamento_seguro.__file__,",
    ),
    (
        "'atualizacoes': 200,",
        "'atualizacoes': 2000,",
    ),
    (
        "'adaptacao': 'full fine-tuning',",
        "'adaptacao': 'fine-tuning ate os blocos do estagio 2; posteriores inativos',",
    ),
    (
        "'estagio': 3, 'normalizacao': 'classifier.norm -> outnorm3',",
        """'estagio': 2,
              'normalizacao': 'LayerNorm fixa escala 1 vies 0; antes de interpolar',
              'plano_treino_sha256': assinatura(planos),
              'alinhamento_degenerado': 'zero prototipos: alinhamento zero, CE ativa, registrar',
              'gpu_driver': subprocess.check_output(
                  ['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],
                  text=True
              ).strip(),""",
    ),
    (
        "'validacao': 'antes e depois; eval; CE e Dice pseudo-FG; isval=False',",
        "'validacao': 'passos 0, 500, 1000, 1500 e 2000; pseudo-mascaras; isval=False',",
    ),
    (
        "'checkpoint_a_cada': 25,",
        "'checkpoint_a_cada': 50,",
    ),
    (
        "f'eft_{variante}_piloto_{agora()}'",
        "f'eft_tiny_penultima_ln_2000_{agora()}'",
    ),
    (
        "shutil.copy2(dados / 'planos.json', pasta / 'planos.json')",
        "gravar_json(pasta / 'planos.json', planos)",
    ),
    (
        "        modelo = construir_modelo(variante, repo)",
        "        modelo = construir_modelo(variante, repo)\n"
        "        estado_align = instalar(modelo)",
    ),
    (
        "0 <= passo <= 200",
        "0 <= passo <= 2000",
    ),
    (
        "        modelo.train()\n        for numero in range(passo + 1, 201):",
        """
        def marco(n):
            if n == 0 or n % 500 != 0:
                return

            destino = pasta / f'checkpoint_{n:04d}.pt'
            if not destino.exists():
                try:
                    os.link(pasta / 'checkpoint_latest.pt', destino)
                except OSError:
                    tmp = destino.with_suffix('.tmp')
                    shutil.copy2(pasta / 'checkpoint_latest.pt', tmp)
                    os.replace(tmp, destino)

            arquivo = pasta / f'validacao_{n:04d}.json'
            if not arquivo.exists():
                rng = estado_rng()
                try:
                    v = avaliar(modelo, dados, planos['validacao'], arquivo)
                finally:
                    restaurar_rng(rng)
                print(
                    f"Validação {n}: CE={v['ce_media']:.6f}; "
                    f"Dice pseudo-FG={v['dice_pseudo_fg_medio']:.6f}",
                    flush=True,
                )

        marco(passo)
        modelo.train()
        for numero in range(passo + 1, 2001):
""",
    ),
    (
        "            otimizador.zero_grad(set_to_none=True)",
        "            otimizador.zero_grad(set_to_none=True)\n"
        "            estado_align['ultimo'] = None",
    ),
    (
        "'regiao': r['regiao'], 'ce': ce.detach().item(),",
        "'regiao': r['regiao'], "
        "'excecao_alinhamento': estado_align['ultimo'], "
        "'ce': ce.detach().item(),",
    ),
    (
        "/200 |",
        "/2000 |",
    ),
    (
        "            if numero % 25 == 0:",
        "            if numero % 50 == 0:",
    ),
    (
        "                salvar_checkpoint(pasta, modelo, otimizador, numero, config, baseline)",
        "                salvar_checkpoint(pasta, modelo, otimizador, numero, config, baseline)\n"
        "                marco(numero)",
    ),
    (
        "final = avaliar(modelo, dados, planos['validacao'], pasta / 'validacao_final.json')",
        "final = json.loads((pasta / 'validacao_2000.json').read_text())\n"
        "        gravar_json(pasta / 'validacao_final.json', final)",
    ),
]

for antes, depois in alteracoes:
    treino = trocar(treino, antes, depois)

ast.parse(modelo)
ast.parse(treino)

arquivos = {
    "eft_modelo.py": modelo,
    "eft_treino.py": treino,
    "eft_dados.py": (BASE / "eft_dados.py").read_text(),
    "eft_alinhamento_seguro.py": CORRECAO.read_text(),
}

PACOTE.mkdir(exist_ok=True)

for nome, conteudo in arquivos.items():
    destino = PACOTE / nome
    if destino.exists():
        assert destino.read_text() == conteudo, (
            f"Já existe outra versão: {destino}"
        )
    else:
        destino.write_text(conteudo)

for caminho in (REPO, PACOTE):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

for nome in (
    "eft_treino", "eft_modelo", "eft_dados", "eft_alinhamento_seguro",
):
    sys.modules.pop(nome, None)

importlib.invalidate_caches()
from eft_treino import executar_piloto

print("Experimento preparado:", PACOTE)
print("Pesos iniciais: checkpoint pré-treinado, sem carregar o EFT anterior.")
print("Nenhum treinamento iniciado.")

In [ ]:
from pathlib import Path
import ast
import sys
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
ORIGINAL = REPO / "exp/eft_penultima_2000_v1"
PACOTE = REPO / "exp/eft_penultima_5000_v1"

fonte = ORIGINAL / "eft_treino.py"
assert fonte.is_file(), "Execute primeiro a célula 1 de preparação."

codigo = fonte.read_text()

alteracoes = [
    (
        "eft_tiny_2000_v2_20261011T204725195685Z/planos.json",
        "eft_tiny_5000_20261011T224215171455Z/planos.json",
    ),
    (
        "len(planos['treino']) == 2000",
        "len(planos['treino']) == 5000",
    ),
    ("'atualizacoes': 2000,", "'atualizacoes': 5000,"),
    ("0 <= passo <= 2000", "0 <= passo <= 5000"),
    ("range(passo + 1, 2001)", "range(passo + 1, 5001)"),
    ("/2000 |", "/5000 |"),
    ("validacao_2000.json", "validacao_5000.json"),
    (
        "f'eft_tiny_penultima_ln_2000_{agora()}'",
        "f'eft_tiny_penultima_ln_5000_{agora()}'",
    ),
    (
        "'validacao': 'passos 0, 500, 1000, 1500 e 2000; pseudo-mascaras; isval=False',",
        "'validacao': 'passo 0 e a cada 500 ate 5000; pseudo-mascaras; isval=False',",
    ),
]

for antes, depois in alteracoes:
    assert codigo.count(antes) == 1, (
        f"Fonte diferente do esperado: {antes}"
    )
    codigo = codigo.replace(antes, depois, 1)

ast.parse(codigo)

arquivos = {"eft_treino.py": codigo}
for nome in (
    "eft_modelo.py",
    "eft_dados.py",
    "eft_alinhamento_seguro.py",
):
    arquivos[nome] = (ORIGINAL / nome).read_text()

PACOTE.mkdir(exist_ok=True)

for nome, conteudo in arquivos.items():
    destino = PACOTE / nome
    if destino.exists():
        assert destino.read_text() == conteudo, (
            f"Já existe outra versão: {destino}"
        )
    else:
        destino.write_text(conteudo)

for caminho in (REPO, PACOTE):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

for nome in (
    "eft_treino", "eft_modelo", "eft_dados", "eft_alinhamento_seguro",
):
    sys.modules.pop(nome, None)

importlib.invalidate_caches()
from eft_treino import executar_piloto

print("Preparado: penúltima camada, LayerNorm fixa, 5.000 passos.")
print("Sequência de episódios: a mesma do treino anterior de 5.000.")
print("Nenhum treinamento iniciado.")

In [ ]:
DADOS = REPO / (
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z"
)

RETOMAR = None

RESULTADOS = executar_piloto(
    dados=DADOS,
    variante="tiny",
    retomar=RETOMAR,
    repo=REPO,
)

print("RESULTADOS PENÚLTIMA:", RESULTADOS)

In [ ]:
from pathlib import Path
import ast
import sys
import os
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA_VAL = REPO / "exp/eft_validacao_polipos_v1"
os.chdir(REPO)

codigo = (PASTA_VAL / "validar_polipos.py").read_text()

for antes, depois in (
    (
        "resumo_piloto['atualizacoes'] == 200",
        "resumo_piloto['atualizacoes'] == 2000",
    ),
    (
        "ckpt_eft = piloto / 'checkpoint_latest.pt'",
        "ckpt_eft = piloto / 'checkpoint_2000.pt'",
    ),
    (
        "salvo['passo'] == 200 and",
        "salvo['passo'] == 2000 and",
    ),
    (
        "f'eft_{variante}_validacao_polipos_{stamp()}'",
        "f'eft_{variante}_referencia_driver_atual_{stamp()}'",
    ),
):
    assert codigo.count(antes) == 1, antes
    codigo = codigo.replace(antes, depois, 1)

ast.parse(codigo)
(PASTA_VAL / "validar_referencia_atual.py").write_text(codigo)

if str(PASTA_VAL) in sys.path:
    sys.path.remove(str(PASTA_VAL))
sys.path.insert(0, str(PASTA_VAL))

importlib.invalidate_caches()
sys.modules.pop("validar_referencia_atual", None)
from validar_referencia_atual import executar as avaliar_referencia

# Em caso de interrupção, informe a pasta impressa por esta avaliação.
RETOMAR_REFERENCIA = None

REFERENCIA_ATUAL = avaliar_referencia(
    piloto=REPO / (
        "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"
    ),
    retomar=RETOMAR_REFERENCIA,
)

print("REFERÊNCIA NO DRIVER ATUAL:", REFERENCIA_ATUAL)